In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
import os
from langchain.chat_models import init_chat_model
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-3.7-flash")
response = model.invoke("hi,how are you?")
response

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content=[{'type': 'text', 'text': "Hello! I'm doing great, thank you for asking. How are you doing today? How can I help you?", 'extras': {'signature': 'EoYFCoMFAWkUfRNBghwr2kkTfMEDpAAqsg4/QVYmd4Z1CimxpXbC0u4nWS6PJ/2r0uRWxKSSTpS0G+mAKIj6o/UE9gvjUqWZOqFRJ/utqFR2VFO7CyQEJUQlF5V286Lj+tZlKYEK74tm6++RlRqrT+Ovf7E/R4J/9rTFno//w5M/ovavB4db/fDzXRdocY+HtIN+e/gGgmv922vhtockEXdhHlaNJhE5wc+H6+GJI5JEfHSp4WKHoY/x+CNFo3W5JtekmZpnK6yNiiC2RBm4r2nr71Ms8oS+wIU6qiQcQsO5O8qjd9rMoNYMVHMgh0Fv6sWYCRUBF2LtIwGe2BnfzICofRGUJyCVaaNokpcDEsEt0oufy4MJyEWwSRxNDb2Y+jGLVAdsTMk+tSjP569NPiRRUaFoQdPMnW4L/0ggx8fs5vfDaCYjWiduaNbeK0H+/3Ldv3Ue31T4b1GGlT3cYQ+ejkavGHDg0ty49CM1Rb1j+6QZ1igvv9Is79mLyNwrbK9j8d9qrU/x2od97Ws38nIu0pg1I2JHW51flbNpz+2VGpO833SY/+OfYVX/2BZO9IfnIXmzU9YYJlwjeG4NnLkOplRA9ssqRPJYA8+B8gHw280pzANH7WfnLwXXU3ZBPhVu+nBPWIVII3I07U+fs+pRYWgpgvbxSprCEvlYU7c6VX7rJzfcV8Ja7UFmh74127hjKDeIBIzLpZrefqitVkycAY7OmWL6XTRKHf71TV5kl0nPT7usewQUkXCjVtZwbCwZ+qOKby6ZhNXtjrUW40w8Br22sBI33FXetnv59pYwOJQQxlGDo6sPfxKGC4vO3fAxl

In [5]:
from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """get the weather for a given location"""
    return f"Its Sunny in {location} today"
model_with_tools = model.bind_tools([get_weather])


In [11]:
response = model_with_tools.invoke("hi,how are you? and what is the weather in New York?")
response

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_2651010': 'EtsCCtgCAWkUfRPbUGXT/yhWGxg518BN0WFhxIQqbnSXFaPeGw/j46OYOVLpqwD5/9XfTbntOkZlwgi34JaZQIMhr+b4BVWeTc5JhrEW07/NJV8PQvZhObhZXlBd8/FJLqEEvG+fKTfraJ3Ii1KifsIMjF3kJtsq6ZVjnC26k1bcdunT4L/n8nk2X2Tqd8MQzYtV22bDkONgDYgvD6V2y30mGT+XTqQdTlEdeHpcGX3WMrwmmdmearIc42/tCvDG16bGolZ311p9Y5DOa93aQSoNZOAt8AfKOgeZZfU1DUIYoolRwCkFm4KjgjmPbKgsX2qZqP3itUuA6h5Jd7/XhBYp9WoTXBx3im1Y0MW/xiy+hrjWl68CCoveorzU4Yr4aa4FLAsGyFA+MidXFcx/X7AAXdSvRoLqF5T2qBwmqqD08A37EOY4oZWZyiEimuAd8T6Fd8sGHM4/WMbTsqc='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.7-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0c246-cb1c-7452-b74b-cd82ba2a679c-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_2651010', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

In [10]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]

ai_msg = model_with_tools.invoke(messages)

messages.append(ai_msg)


# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)


# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)

print(final_response.text)

# "The current weather in Boston is 72°F and sunny."

The weather in Boston today is sunny.
